# Task 2 — Heart Disease Prediction System

**Live deployment:** PENDING — paste your deployed Streamlit URL here after publishing.

Adapted from the instructor’s Titanic prediction system. Educational use only; not a medical diagnosis tool. Dataset: UCI Heart Disease (Cleveland).

## Step 1 — Import libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Step 2 — Collect and load UCI Heart Disease data
The notebook first looks for `processed.cleveland.data` locally; otherwise it attempts to download the official UCI source. Internet access is needed for the first download.

In [2]:
DATA_PATH = Path("processed.cleveland.data")
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
if not DATA_PATH.exists():
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
data = pd.read_csv(DATA_PATH, header=None, names=columns, na_values="?")
print("Dataset shape:", data.shape)
display(data.head())

Dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## Step 3 — Understand and preprocess data
The target `num` is 0 for no diagnosed disease and 1–4 for presence. We map it to binary 0/1. Only four input attributes are used: age, resting blood pressure, cholesterol, and maximum heart rate. Missing values are handled inside a training-only pipeline to avoid leakage.

In [3]:
print(data.info())
print("Missing values:\n", data.isna().sum())
FEATURES = ["age", "trestbps", "chol", "thalach"]
X = data[FEATURES].copy()
y = (data["num"] > 0).astype(int)
print("Class counts:\n", y.value_counts())
display(X.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 33.3 KB
None
Missing values:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
num         0
d

,age,trestbps,chol,thalach
count,303.000000,303.000000,303.000000,303.000000
mean,54.438944,131.689769,246.693069,149.607261
std,9.038662,17.599748,51.776918,22.875003
min,29.000000,94.000000,126.000000,71.000000
25%,48.000000,120.000000,211.000000,133.500000
50%,56.000000,130.000000,241.000000,153.000000
75%,61.000000,140.000000,275.000000,166.000000
max,77.000000,200.000000,564.000000,202.000000


## Step 4 — Feature extraction and label encoding
Four numeric features are selected. The binary target is already encoded numerically, so no categorical LabelEncoder is required.

In [4]:
print("Selected features:", FEATURES)
print("Output labels: 0 = no disease, 1 = disease present")

Selected features: ['age', 'trestbps', 'chol', 'thalach']
Output labels: 0 = no disease, 1 = disease present


## Step 5 — Training phase: split and train Support Vector Classifier

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("svc", SVC(kernel="rbf", class_weight="balanced"))])
model.fit(X_train, y_train)
print("Train:", len(X_train), "Test:", len(X_test))

Train: 242 Test: 61


## Step 6 — Save trained model

In [6]:
joblib.dump({"pipeline": model, "features": FEATURES}, "heart_disease_model.joblib")
print("Saved heart_disease_model.joblib")

Saved heart_disease_model.joblib


## Step 7 — Testing phase and evaluation

In [7]:
loaded = joblib.load("heart_disease_model.joblib")["pipeline"]
y_pred = loaded.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

Test accuracy: 0.7213
Confusion matrix:
 [[23 10]
 [ 7 21]]
              precision    recall  f1-score   support

           0       0.77      0.70      0.73        33
           1       0.68      0.75      0.71        28

    accuracy                           0.72        61
   macro avg       0.72      0.72      0.72        61
weighted avg       0.73      0.72      0.72        61



## Step 8 — Application phase: predict a new example
Illustrative example only, not medical advice.

In [8]:
example = pd.DataFrame([[54, 130, 240, 150]], columns=FEATURES)
prediction = loaded.predict(example)[0]
print("Example prediction:", "Disease present" if prediction == 1 else "No disease detected")

Example prediction: No disease detected


## Step 9 — Feedback phase
Collect user feedback about input clarity and unexpected predictions. Potential improvements: clinical validation, larger representative datasets, model calibration, fairness analysis, and professional review. Retraining should be performed only with appropriate validated data.

## Step 10 — Cloud deployment
1. Run all cells so `heart_disease_model.joblib` is generated.
2. Upload `app.py`, `heart_disease_model.joblib`, and `requirements.txt` to a GitHub repository.
3. Create a Streamlit Community Cloud application pointing to `app.py`.
4. Test predictions, then paste its live URL at the top of this notebook.

**Submission:** notebook, UCI dataset, trained model file, and deployment URL.